In [8]:
import pandas as pd
import json
import numpy as np

In [4]:
def process_qualt5_res_individual(_path: str, _cutoff: int, _readability_metric=''):
    _quality_df = pd.read_csv(_path)
    if('readability' in _path):
        _quality_df = _quality_df.rename(columns={'readability_score': 'quality'})
        _quality_df = _quality_df.query('readability_metric==@_readability_metric')
    else:
        _quality_df.quality = _quality_df.quality.apply(lambda x: np.exp(x))
    qualt5_individual_res = _quality_df.query('rank<@_cutoff')[['qid', 'docno', 'rank', 'quality']]

    
    qualt5_individual_res = qualt5_individual_res[qualt5_individual_res.quality != -1]
    qualt5_individual_res_groupby_qid = qualt5_individual_res.groupby(['qid'])['quality']
    _doc_qual_max = dict(zip(qualt5_individual_res_groupby_qid.max().index.astype('str').values, qualt5_individual_res_groupby_qid.max().values))
    _doc_qual_avg = dict(zip(qualt5_individual_res_groupby_qid.mean().index.astype('str').values, qualt5_individual_res_groupby_qid.mean().values))
    return _doc_qual_max, _doc_qual_avg

def process_qualt5_res_integrated(_path: str, _readability_metric=''):
    _quality_df = pd.read_csv(_path)
    if('readability' in _path):
        _quality_df = _quality_df.rename(columns={'score': 'quality'})
        _quality_df = _quality_df.query('readability_metric==@_readability_metric')
    else:
        _quality_df.quality = _quality_df.quality.apply(lambda x: np.exp(x))
        
    qualt5_integrated_res = _quality_df[['qid', 'docno', 'quality']]

    _doc_qual_itg = dict(zip(qualt5_integrated_res.qid.astype('str').values, qualt5_integrated_res.quality.values))
    return _doc_qual_itg

In [61]:
import copy

_k = 3
_ret = 'mt5'
_task = 'dl'
target_metric = 'f1'

if(_task=='nq'):
    _dataset_dev, _dataset_test, _prefix, _suffix = 'nq_dev', ['nq_test'], 'short', 'concise'
elif(_task=='dl'):
    _dataset_dev, _dataset_test, _prefix, _suffix = 'dev_small', ['19', '20'], 'random', 'prompt1'

In [50]:
# loading dev data
f = open(f'../../rag_utility/eval_results/{_prefix}_answers_0shot_1calls_0_0_bm25_dl_{_dataset_dev}_{_suffix}_eval.json')
zero_evals = json.load(f)
f.close()

f = open(f'../../rag_utility/eval_results/{_prefix}_answers_{_k}shot_1calls_1_0_{_ret}_dl_{_dataset_dev}_{_suffix}_eval.json')
k_evals = json.load(f)
f.close()

f = open(f'../../rag_utility/gen_results/{_prefix}_answers_{_k}shot_1calls_1_0_{_ret}_dl_{_dataset_dev}_{_suffix}.json')
k_gens = json.load(f)
f.close()

f = open(f'../perplexity_eval/log_prob_temp_res/full_context_with_query/{_dataset_dev}_{_ret}_{_k}.json')
perpC = json.load(f)
f.close()

readability_res_path = f'../readability_eval/readability_res/individual_readability_{_dataset_dev}_{_ret}_top_10.csv'
readability_res_path_itg = f'../readability_eval/readability_res/integrated_readability_{_dataset_dev}_{_ret}_top_{_k}.csv'
available_metrics = pd.read_csv(readability_res_path).readability_metric.unique()
readability_max_dict, readability_avg_dict, readability_itg_dict = {}, {}, {}
for _r_m in available_metrics:
    _r_m_max, _r_m_avg = process_qualt5_res_individual(readability_res_path, _k, _r_m)
    _r_m_itg = process_qualt5_res_integrated(readability_res_path_itg, _r_m)
    readability_max_dict.update({_r_m: copy.deepcopy(_r_m_max)})
    readability_avg_dict.update({_r_m: copy.deepcopy(_r_m_avg)})
    readability_itg_dict.update({_r_m: copy.deepcopy(_r_m_itg)})

doc_qual_max, doc_qual_avg = process_qualt5_res_individual(f'../qualt5_eval/quality_res/{_ret}_{_dataset_dev}.csv', _k)
doc_qual_itg = process_qualt5_res_integrated(f'../qualt5_eval/quality_res/{_ret}_{_dataset_dev}_integrated_{_k}.csv')

qpp_df = pd.read_csv(f'./precomputed_qpps/{_ret}_{_k}_combined_qpp_{_dataset_dev}.csv')

dev_res = qpp_df[['qid', 'query']].drop_duplicates().copy()

# Expand the dataframe for the convenience of analysis
for qpp_name in qpp_df.qpp_method.unique():
    value_dict = dict(zip(qpp_df[qpp_df.qpp_method==qpp_name]['qid'], qpp_df[qpp_df.qpp_method==qpp_name]['qpp_estimate']))
    dev_res[qpp_name] = dev_res.qid.apply(lambda _qid: value_dict[_qid])

if(_task=='nq'):
    base_f1_dict = {item[0]: item[1]['0']['0']['F1'] for item in zero_evals.items()}
    f1_dict = {item[0]: item[1]['0']['0']['F1'] for item in k_evals.items()}
    # em_dict = {item[0]: item[1]['0']['0']['EM'] for item in k_evals.items()}
    kshot_prob_dict = {item[0]: np.mean(eval(item[1]['0']['0']['probs'])) for item in k_gens.items()}
elif(_task=='dl'):
    base_f1_dict = {item[0]: item[1]['0']['0']['qrel_1']['f1']['max'] for item in zero_evals.items() if ('0' in item[1].keys())}
    f1_dict = {item[0]: item[1]['0']['0']['qrel_1']['f1']['max'] for item in k_evals.items() if ('0' in item[1].keys())}
    kshot_prob_dict = {item[0]: np.mean(eval(item[1]['0']['0']['probs'])) for item in k_gens.items() if ('0' in item[1].keys())}

dev_res = dev_res[dev_res.qid.astype('str').isin(f1_dict.keys())]
dev_res['f1'] = dev_res.qid.apply(lambda x: f1_dict[str(x)])
dev_res['utility'] = dev_res.qid.apply(lambda x: f1_dict[str(x)]-base_f1_dict[str(x)])
# dev_res['em'] = dev_res.qid.apply(lambda x: em_dict[str(x)])
dev_res['prob(k)'] = dev_res.qid.apply(lambda x: kshot_prob_dict[str(x)])
dev_res['perpC'] = dev_res.qid.apply(lambda x: perpC[str(x)])
dev_res['max(docQual)'] = dev_res.qid.apply(lambda x: doc_qual_max[str(x)])
dev_res['avg(docQual)'] = dev_res.qid.apply(lambda x: doc_qual_avg[str(x)])
dev_res['itg(docQual)'] = dev_res.qid.apply(lambda x: doc_qual_itg[str(x)])
print(dev_res.shape)
dev_res = dev_res[dev_res.qid.astype('str').isin(readability_itg_dict['Spache'].keys())]
print(dev_res.shape)

readability_cols = []
for _r_m in available_metrics:
    readability_cols += [f'max({_r_m})', f'avg({_r_m})', f'itg({_r_m})']
    print(_r_m, len(readability_max_dict[_r_m]))

    for qid in dev_res.qid.unique():
        if qid not in readability_max_dict[_r_m].keys():
            # print(qid)
            readability_max_dict[_r_m].update({str(qid): -1})
            readability_avg_dict[_r_m].update({str(qid): -1})
        if qid not in readability_itg_dict[_r_m].keys():
            readability_itg_dict[_r_m].update({str(qid): -1})
    
    dev_res[f'max({_r_m})'] = dev_res.qid.apply(lambda x: readability_max_dict[_r_m][str(x)])
    dev_res[f'avg({_r_m})'] = dev_res.qid.apply(lambda x: readability_avg_dict[_r_m][str(x)])
    dev_res[f'itg({_r_m})'] = dev_res.qid.apply(lambda x: readability_itg_dict[_r_m][str(x)])

dev_res = dev_res.drop(columns=[col for col in readability_cols if (dev_res[col] == -1).any()])

dev_res = dev_res.dropna(axis='index')
dev_res.head(3)


FileNotFoundError: [Errno 2] No such file or directory: '../perplexity_eval/log_prob_temp_res/full_context_with_query/dev_small_bm25_3.json'

In [75]:
# loading test data

import numpy as np

def tool_for_aggregating_dl_performance(x):
    # the same as performLoader
    scores = []
    for answer_eval in x[1]['0'].values():
        scores.append(max(answer_eval['qrel_2']['f1']['max'], answer_eval['qrel_3']['f1']['max']))
    return np.mean(scores)

zero_evals, k_evals, k_gens, perpC  = {}, {}, {}, {}

_calls = 5 if _task=='dl' else 1

for _d in _dataset_test:
    f = open(f'../../rag_utility/eval_results/{_prefix}_answers_0shot_{_calls}calls_0_0_bm25_dl_{_d}_{_suffix}_eval.json')
    zero_evals.update(json.load(f))
    f.close()
    
    f = open(f'../../rag_utility/eval_results/{_prefix}_answers_{_k}shot_{_calls}calls_1_0_{_ret}_dl_{_d}_{_suffix}_eval.json')
    print(f'../../rag_utility/eval_results/{_prefix}_answers_{_k}shot_{_calls}calls_1_0_{_ret}_dl_{_d}_{_suffix}_eval.json')
    k_evals.update(json.load(f))
    f.close()
    
    f = open(f'../../rag_utility/gen_results/{_prefix}_answers_{_k}shot_{_calls}calls_1_0_{_ret}_dl_{_d}_{_suffix}.json')
    k_gens.update(json.load(f))
    f.close()

if(_task == 'dl'):
    _d = 'dl'
else:
    _d = 'nq_test'
    
f = open(f'../perplexity_eval/log_prob_temp_res/full_context_with_query/{_d}_{_ret}_{_k}.json')
perpC.update(json.load(f))
f.close()

readability_res_path = f'../readability_eval/readability_res/individual_readability_{_d}_{_ret}_top_10.csv'
readability_res_path_itg = f'../readability_eval/readability_res/integrated_readability_{_d}_{_ret}_top_{_k}.csv'
available_metrics = pd.read_csv(readability_res_path).readability_metric.unique()
readability_max_dict, readability_avg_dict, readability_itg_dict = {}, {}, {}
for _r_m in available_metrics:
    _r_m_max, _r_m_avg = process_qualt5_res_individual(readability_res_path, _k, _r_m)
    _r_m_itg = process_qualt5_res_integrated(readability_res_path_itg, _r_m)
    readability_max_dict.update({_r_m: copy.deepcopy(_r_m_max)})
    readability_avg_dict.update({_r_m: copy.deepcopy(_r_m_avg)})
    readability_itg_dict.update({_r_m: copy.deepcopy(_r_m_itg)})

doc_qual_max, doc_qual_avg = process_qualt5_res_individual(f'../qualt5_eval/quality_res/{_ret}_{_d}.csv', _k)
doc_qual_itg = process_qualt5_res_integrated(f'../qualt5_eval/quality_res/{_ret}_{_d}_integrated_{_k}.csv')

qpp_df = pd.read_csv(f'./precomputed_qpps/{_ret}_{_k}_combined_qpp_{_d}.csv')

test_res = qpp_df[['qid', 'query']].drop_duplicates().copy()

# Expand the dataframe for the convenience of analysis
for qpp_name in qpp_df.qpp_method.unique():
    value_dict = dict(zip(qpp_df[qpp_df.qpp_method==qpp_name]['qid'], qpp_df[qpp_df.qpp_method==qpp_name]['qpp_estimate']))
    test_res[qpp_name] = test_res.qid.apply(lambda _qid: value_dict[_qid])

if(_task=='nq'):
    base_f1_dict = {item[0]: item[1]['0']['0']['F1'] for item in zero_evals.items()}
    f1_dict = {item[0]: item[1]['0']['0']['F1'] for item in k_evals.items()}
    # em_dict = {item[0]: item[1]['0']['0']['EM'] for item in k_evals.items()}
    kshot_prob_dict = {item[0]: np.mean(eval(item[1]['0']['0']['probs'])) for item in k_gens.items()}
elif(_task=='dl'):
    base_f1_dict = {item[0]: tool_for_aggregating_dl_performance(item) for item in zero_evals.items() if ('0' in item[1].keys())}
    f1_dict = {item[0]: tool_for_aggregating_dl_performance(item) for item in k_evals.items() if ('0' in item[1].keys())}
    kshot_prob_dict = {item[0]: np.mean(eval(item[1]['0']['0']['probs'])) for item in k_gens.items() if ('0' in item[1].keys())}

test_res = test_res[test_res.qid.astype('str').isin(f1_dict.keys())]
test_res['f1'] = test_res.qid.apply(lambda x: f1_dict[str(x)])
test_res['utility'] = test_res.qid.apply(lambda x: f1_dict[str(x)]-base_f1_dict[str(x)])
# test_res['em'] = test_res.qid.apply(lambda x: em_dict[str(x)])
test_res['prob(k)'] = test_res.qid.apply(lambda x: kshot_prob_dict[str(x)])
test_res['perpC'] = test_res.qid.apply(lambda x: perpC[str(x)])
test_res['max(docQual)'] = test_res.qid.apply(lambda x: doc_qual_max[str(x)])
test_res['avg(docQual)'] = test_res.qid.apply(lambda x: doc_qual_avg[str(x)])
test_res['itg(docQual)'] = test_res.qid.apply(lambda x: doc_qual_itg[str(x)])

print(test_res.shape)
test_res = test_res[test_res.qid.astype('str').isin(readability_itg_dict['Spache'].keys())]
print(test_res.shape)

readability_cols = []
for _r_m in available_metrics:
    readability_cols += [f'max({_r_m})', f'avg({_r_m})', f'itg({_r_m})']
    print(_r_m, len(readability_max_dict[_r_m]))

    for qid in test_res.qid.unique():
        if qid not in readability_max_dict[_r_m].keys():
            # print(qid)
            readability_max_dict[_r_m].update({str(qid): -1})
            readability_avg_dict[_r_m].update({str(qid): -1})
        if qid not in readability_itg_dict[_r_m].keys():
            readability_itg_dict[_r_m].update({str(qid): -1})
    
    test_res[f'max({_r_m})'] = test_res.qid.apply(lambda x: readability_max_dict[_r_m][str(x)])
    test_res[f'avg({_r_m})'] = test_res.qid.apply(lambda x: readability_avg_dict[_r_m][str(x)])
    test_res[f'itg({_r_m})'] = test_res.qid.apply(lambda x: readability_itg_dict[_r_m][str(x)])

dropped_columns = [col for col in readability_cols if (test_res[col] == -1).any()]
test_res = test_res.drop(columns=[col for col in readability_cols if (test_res[col] == -1).any()])
readability_cols = [e for e in readability_cols if e not in dropped_columns]
test_res.head(3)

../../rag_utility/eval_results/random_answers_3shot_5calls_1_0_mt5_dl_19_prompt1_eval.json
../../rag_utility/eval_results/random_answers_3shot_5calls_1_0_mt5_dl_20_prompt1_eval.json
(97, 15)
(97, 15)
Dale Chall 11
Spache 11
Flesch-Kincaid 11
Flesch 11
Gunning Fog 11
Coleman Liau 11
ARI 11
Linsear Write 11
SMOG 0


,qid,query,nqc,maxScore,spatial,a_ratio,bertQPP,bertQPP(QV),f1,utility,prob(k),perpC,max(docQual),avg(docQual),itg(docQual)
0,1030303,who is aziz hashim,2.385411e-06,0.998086,2836.811279,6.409228,0.453741,0.317681,0.985931,0.428543,-0.051232,-2.083984,0.847861,0.760612,0.842084
1,1037496,who is rep scalise,2.127431e-07,0.995001,3138.243408,3.228394,0.672983,0.212220,0.788641,0.116377,-0.159821,-0.860840,0.870514,0.848020,0.643131
2,1037798,who is robert gray,9.612573e-06,0.995599,2869.693115,20.823040,0.529355,0.302883,0.456429,-0.101195,-0.195101,-2.433594,0.748965,0.617111,0.606531


In [76]:
from scipy import stats

In [77]:
import pandas as pd

df_content = []

df_content.append(['nqc', stats.spearmanr(test_res.nqc, test_res[target_metric])[0], stats.kendalltau(test_res.nqc, test_res[target_metric])[0]])
df_content.append(['maxScore', stats.spearmanr(test_res.maxScore, test_res[target_metric])[0], stats.kendalltau(test_res.maxScore, test_res[target_metric])[0]])
df_content.append(['denseQPP', stats.spearmanr(test_res.spatial, test_res[target_metric])[0], stats.kendalltau(test_res.spatial, test_res[target_metric])[0]])
df_content.append(['aPairRatio', stats.spearmanr(test_res.a_ratio, test_res[target_metric])[0], stats.kendalltau(test_res.a_ratio, test_res[target_metric])[0]])
df_content.append(['bertQPP', stats.spearmanr(test_res.bertQPP, test_res[target_metric])[0], stats.kendalltau(test_res.bertQPP, test_res[target_metric])[0]])

df_content.append(['perpC', stats.spearmanr(test_res.perpC, test_res[target_metric])[0], stats.kendalltau(test_res.perpC, test_res[target_metric])[0]])
df_content.append(['max(docQual)', stats.spearmanr(test_res['max(docQual)'], test_res[target_metric])[0], stats.kendalltau(test_res['max(docQual)'], test_res[target_metric])[0]])
df_content.append(['avg(docQual)', stats.spearmanr(test_res['avg(docQual)'], test_res[target_metric])[0], stats.kendalltau(test_res['avg(docQual)'], test_res[target_metric])[0]])
df_content.append(['itg(docQual)', stats.spearmanr(test_res['itg(docQual)'], test_res[target_metric])[0], stats.kendalltau(test_res['itg(docQual)'], test_res[target_metric])[0]])

df_content.append(['prob(k)', stats.spearmanr(test_res['prob(k)'], test_res[target_metric])[0], stats.kendalltau(test_res['prob(k)'], test_res[target_metric])[0]])

a1 = pd.DataFrame(df_content, columns=['QPP_Method', 'Spearman', 'Kendall'])
a1.Spearman = a1.Spearman.apply(lambda x: round(x, 4))
a1.Kendall = a1.Kendall.apply(lambda x: round(x, 4))
a1.to_csv('./temp_for_pasting_results/a1.csv', index=False)
a1

,QPP_Method,Spearman,Kendall
0,nqc,-0.0960,-0.0679
1,maxScore,0.0882,0.0558
2,denseQPP,0.4091,0.3076
3,aPairRatio,-0.1466,-0.1061
4,bertQPP,0.1808,0.1284
5,perpC,0.3557,0.2486
6,max(docQual),0.1813,0.1248
7,avg(docQual),0.2586,0.1804
8,itg(docQual),0.0147,0.0000
9,prob(k),0.5770,0.4326


#### Learned Combination -> Linear Regression

In [153]:
from sklearn import linear_model
import itertools
import pathlib

output_content = []

output_path = "./ecir_res/union_output_v1.csv"

# define features
for use_postgen, pregen_combination in itertools.product([False], ['0', '1', '2', '3', '12', '13', '23', '123', '0']):
    # use_postgen=True
    # pregen_combination = 0
    print(pregen_combination, use_postgen)
    # if(_ret=='e5'):
    #     used_qpp_methods = ['nqc', 'spatial', 'maxScore', 'a_ratio', 'bertQPP']
    # else:
    #     used_qpp_methods = ['nqc', 'maxScore', 'bertQPP']
    used_qpp_methods = ['nqc', 'spatial', 'maxScore', 'a_ratio', 'bertQPP']

    used_reader_centric_predictors = []
    # if('0' in pregen_combination):
    #     used_reader_centric_predictors = []
    if('1' in pregen_combination):
        used_reader_centric_predictors += ['perpC']
    if('2' in pregen_combination):
        used_reader_centric_predictors += ['max(docQual)', 'avg(docQual)', 'itg(docQual)']
    if('3' in pregen_combination):
        used_reader_centric_predictors += readability_cols
    
    # load data
    retr_cen = dev_res[used_qpp_methods].apply(lambda x: np.log(1+x)).values
    retr_cen_test = test_res[used_qpp_methods].apply(lambda x: np.log(1+x)).values
    
    if(used_reader_centric_predictors==[]):
        dev_data = retr_cen
        test_data = retr_cen_test
    else:
        reader_cen = dev_res[used_reader_centric_predictors].values
        dev_data = np.hstack((retr_cen, reader_cen))
        
        reader_cen_test = test_res[used_reader_centric_predictors].values
        test_data = np.hstack((retr_cen_test, reader_cen_test))
    if(use_postgen):
        dev_data = np.hstack((dev_data, dev_res[['prob(k)']].values))
        test_data = np.hstack((test_data, test_res[['prob(k)']].values))

    # best before combination
    best_row = a1[a1.QPP_Method.isin(used_qpp_methods+used_reader_centric_predictors+use_postgen*['prob(k)'])].query('Spearman==Spearman.max()').iloc[0]
    print(f'Best before combination is {best_row.QPP_Method}, rho={best_row.Spearman}, tau={best_row.Kendall}')
    
    # linear regression
    reg = linear_model.LinearRegression()
    reg.fit(dev_data, dev_res[target_metric].values)
    coefs = reg.coef_
    intercept = reg.intercept_
    predictions = (dev_data * coefs).sum(axis=1) + intercept
    
    # print(coefs)
    # print(intercept)
    # print('accuracy on Dev set', stats.spearmanr(predictions, dev_res[target_metric]))
    
    try:
        f = open('./temp_for_pasting_results/weights.json', 'r+', encoding='UTF-8')
        weights = json.load(f)
        f.close()
    except:
        weights = {}
    
    weights.update({f'{_k}-{_ret}-{_task}-{target_metric}': list(coefs)})
    f = open('./temp_for_pasting_results/weights.json', 'w+')
    json.dump(weights, f, indent=4)
    f.close()
    
    predictions_test = ((test_data * coefs).sum(axis=1) + intercept)

    result_rho, result_tau = stats.spearmanr(predictions_test, test_res[target_metric])[0], stats.kendalltau(predictions_test, test_res[target_metric])[0]
    print(f'Accuracy on Test set, rho={result_rho}, tau={result_tau}')
    output_content.append(['GPP' if target_metric=='f1' else 'RPP', _task, _ret, _k, use_postgen, pregen_combination, result_rho, result_tau, best_row.QPP_Method, best_row.Spearman, best_row.Kendall])

temp_output = pd.DataFrame(output_content, columns=['Prediction Name', 'QA Task', 'Retriever', 'Top-Retrieved Docs', 'Use_Postgen', 'Combination_Number', 'Rho', 'Tau', 'Best Single Signal', 'Best Single Rho', 'Best Single Tau'])
csvfile = pathlib.Path(output_path)
temp_output.to_csv(output_path, mode='a', index=False, header=not csvfile.exists())
# deduplicate
x = pd.read_csv(output_path)
x = x.drop_duplicates()
x.to_csv(output_path, index=False)

0 False
Best before combination is bertQPP, rho=0.2113, tau=0.1617
Accuracy on Test set, rho=0.2562469333027465, tau=0.19622755704533276
1 False
Best before combination is bertQPP, rho=0.2113, tau=0.1617
Accuracy on Test set, rho=0.2573831202062929, tau=0.19706081930276978
2 False
Best before combination is bertQPP, rho=0.2113, tau=0.1617
Accuracy on Test set, rho=0.2647523543282786, tau=0.20268199547569707
3 False
Best before combination is bertQPP, rho=0.2113, tau=0.1617
Accuracy on Test set, rho=0.26359423728629877, tau=0.20194472754820086
12 False
Best before combination is bertQPP, rho=0.2113, tau=0.1617
Accuracy on Test set, rho=0.263786243121932, tau=0.20188532122106534
13 False
Best before combination is bertQPP, rho=0.2113, tau=0.1617
Accuracy on Test set, rho=0.26419277570165733, tau=0.20254036449709586
23 False
Best before combination is bertQPP, rho=0.2113, tau=0.1617
Accuracy on Test set, rho=0.2689059300984463, tau=0.20617555961452644
123 False
Best before combination is 

### Random Forest

In [143]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error

# X_train, X_val, y_train, y_val = train_test_split(
#     dev_data, y, dev_data, random_state=42

rf = RandomForestRegressor(
    n_estimators=400,
    criterion='squared_error', # can be squared_error, absolute_error
    max_features='sqrt',
    random_state=42,
    n_jobs=-1
)

rf.fit(dev_data, dev_res[target_metric].values)

RandomForestRegressor(max_features='sqrt', n_estimators=400, n_jobs=-1,
                      random_state=42)

In [144]:
predictions_from_rf = rf.predict(test_data)
result_rho, result_tau = stats.spearmanr(predictions_from_rf, test_res[target_metric])[0], stats.kendalltau(predictions_from_rf, test_res[target_metric])[0]

In [145]:
result_rho, result_tau

(0.22485372025364972, 0.1699408149793209)

In [146]:
from sklearn.model_selection import RandomizedSearchCV
from sklearn.metrics import make_scorer
from scipy.stats import randint

def spearman_corr(y_true, y_pred):
    # Spearmanr returns (correlation, pvalue)
    return stats.spearmanr(y_true, y_pred).correlation

spearman_scorer = make_scorer(spearman_corr, greater_is_better=True)

param_dist = {
    "n_estimators": randint(200, 800),
    "max_depth": randint(3, 30),
    "min_samples_split": randint(2, 20),
    "min_samples_leaf": randint(1, 10),
    # "max_features": ["sqrt", "log2", None]
    "max_features": ["sqrt"]
}

search = RandomizedSearchCV(
    rf,
    param_distributions=param_dist,
    n_iter=40,
    scoring=spearman_scorer,   # <-- key difference
    cv=5,
    n_jobs=-1,
    random_state=42,
    verbose=1 # 2 for printing per-iteration results, 1 for simple display
)

search.fit(dev_data, dev_res[target_metric].values)
print("Best Spearman rho (CV):", search.best_score_)
print("Best params:", search.best_params_)
best_model = search.best_estimator_

Fitting 5 folds for each of 40 candidates, totalling 200 fits
Best Spearman rho (CV): 0.27155104753098297
Best params: {'max_depth': 5, 'max_features': 'sqrt', 'min_samples_leaf': 1, 'min_samples_split': 6, 'n_estimators': 417}


In [138]:
predictions_from_best_rf = best_model.predict(test_data)
result_rho, result_tau = stats.spearmanr(predictions_from_best_rf, test_res[target_metric])[0], stats.kendalltau(predictions_from_best_rf, test_res[target_metric])[0]

In [139]:
result_rho, result_tau

(0.2932030842580639, 0.22384360824763963)

### Correlations between predictions

In [1]:
import pandas as pd
import matplotlib.pyplot as plt

# Example: suppose df has predictor columns
# df = pd.DataFrame({
#     "pred1": [0.1, 0.2, 0.3],
#     "pred2": [0.2, 0.1, 0.4],
#     "pred3": [0.3, 0.4, 0.5],
# })
# Compute correlation matrix
corr = test_res.drop(columns=['qid', 'query', 'f1', 'utility', 'bertQPP(QV)']).corr(method="spearman")

# Plot heatmap using matplotlib
fig, ax = plt.subplots(figsize=(6, 5))
cax = ax.matshow(corr, cmap="coolwarm")

# Add colorbar
fig.colorbar(cax)

# Set ticks and labels
ax.set_xticks(range(len(corr.columns)))
ax.set_yticks(range(len(corr.columns)))
ax.set_xticklabels(corr.columns, rotation=45, ha="left")
ax.set_yticklabels(corr.columns)

plt.title(f'{_task}_{_ret}_{_k}', pad=20)
plt.savefig(f"./plotting/heatmaps/heatmap_{_task}_{_ret}_{_k}_v1.png", format="png", bbox_inches="tight")  
plt.savefig(f"./plotting/heatmaps/heatmap_{_task}_{_ret}_{_k}_v1.pdf", format="pdf", bbox_inches="tight")  
plt.show()

NameError: name 'test_res' is not defined